# 第 6 章:模型组装 —— 嵌入 → 堆叠 Block → lm_head → 绑定权重

前 3 章(第 3-5 章)我们逐个拆解了 minimind 的零件:Config 超参数、RMSNorm、RoPE、Attention、FeedForward、Block。

现在到了**总装车间** —— 把这些零件拼成一台完整的引擎。

本章对应 `model_minimind.py` 的 **210-310 行**:
- `MiniMindModel`(210-264):backbone,负责 embed → N×Block → final norm
- `MiniMindForCausalLM`(266-310):在 backbone 上加 lm_head + loss

> 本章的核心是**完整 forward pass 的 tensor shape 追踪** —— 用代码逐步打印每一步的形状变化,让整条数据流水线变得透明。
> 
> ⚠️ `generate`(314-345)是第 7 章的主题,本章不教,只在结尾点一下衔接。

## 环境准备

先导入 minimind 的组件,构造一个标准的 64M 模型。后续所有代码都基于这个模型实例。

注意:这里用**随机初始化**(不加载预训练权重)—— 因为本章关心的是**结构**,不是权重内容。

In [ ]:
import sys, torch, torch.nn as nn
sys.path.insert(0, '/home/minimind')

from model.model_minimind import (
    MiniMindConfig, MiniMindForCausalLM, MiniMindModel,
    MiniMindBlock, RMSNorm, FeedForward, MOEFeedForward,
    precompute_freqs_cis,
)

# 标准 minimind 配置:768 维, 8 层, 6400 词表
config = MiniMindConfig()
model = MiniMindForCausalLM(config).eval()

print(f"hidden_size    = {config.hidden_size}")        # 768
print(f"num_hidden_layers = {config.num_hidden_layers}")  # 8
print(f"vocab_size     = {config.vocab_size}")         # 6400
print(f"head_dim       = {config.head_dim}")           # 96
print(f"intermediate_size = {config.intermediate_size}") # 2432
print(f"tie_word_embeddings = {config.tie_word_embeddings}")  # True

&nbsp;

---

## 6.1 Embedding 层:从 token id 到向量

一切的起点。`input_ids` 是整数序列,模型需要把它们变成浮点向量才能做矩阵运算。

这是 `MiniMindModel.__init__` 的第一行(201 行):

```python
self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
#                               6400                        768
```

`nn.Embedding` 本质上是一个**查表(lookup table)**:
- 内部维护一个权重矩阵,形状 `(vocab_size, hidden_size)` = `(6400, 768)`
- 输入一个 token id,返回对应行(768 维向量)
- 这个矩阵是**可学习的** —— 每个 token 的向量表示在训练中被不断调整

> 为什么叫 embed?「embedding」= 嵌入,把离散的整数 id「嵌入」到连续的高维空间中。语义相近的 token 最终会获得相近的向量。

下面用代码观察 `embed_tokens` 的结构和 token id → 向量的 shape 变化:

In [ ]:
# 观察 embed_tokens 的结构
# nn.Embedding 本质是查表:权重矩阵 (vocab_size, hidden_size)
embed = model.model.embed_tokens
print(f"类型: {type(embed).__name__}")           # Embedding
print(f"权重 shape: {embed.weight.shape}")        # (6400, 768)
print(f"权重参数量: {embed.weight.numel():,}")    # 4,915,200

# 模拟 token id → 向量
input_ids = torch.tensor([[1, 5310, 2863, 2]])   # (1, 4) — 假装是 [bos, 你好, ,, eos]
print(f"\ninput_ids shape: {input_ids.shape}")    # (1, 4)
print(f"input_ids: {input_ids.tolist()}")

# 查表:id → 向量
hidden = embed(input_ids)
print(f"hidden shape: {hidden.shape}")            # (1, 4, 768) — (batch, seq_len, hidden_size)
# (1, 4) → (1, 4, 768):每个 token id 变成一个 768 维向量

关键 shape 变化:

```
input_ids  (batch, seq_len)         = (1, 4)
    │ embed_tokens (查表)
    ▼
hidden     (batch, seq_len, hidden) = (1, 4, 768)
```

为什么 embedding 可学习?`nn.Embedding` 的权重是一个 `nn.Parameter`(requires_grad=True),和 Linear 层的权重一样会被梯度更新。训练时,模型会学会给每个 token 分配一个「好」的向量 —— 让语义相关的 token 向量彼此靠近。

> 注意:这里的 `dropout` 层(`202` 行)在训练时给 embedding 加随机噪声做正则化。推理时(eval 模式)是 no-op。

## 6.2 堆叠 N 个 Block:backbone 的核心

一个 Transformer 不是一层,而是 **N 层 Block 串联**。`MiniMindModel` 用 `nn.ModuleList` 把 8 个 `MiniMindBlock` 堆起来:

```python
# model_minimind.py:217
self.layers = nn.ModuleList([
    MiniMindBlock(l, config) for l in range(self.num_hidden_layers)
])
```

每个 block 的输入输出都是 `(batch, seq_len, 768)` —— 像一条管道,数据流过去形状不变,但**内容变了**(每层都在提取不同层次的语义信息)。

第 5 章已经拆解过单个 Block 内部(RMSNorm → Attention → 残差 → RMSNorm → FFN → 残差),这里只关注**怎么把它们串起来**。

> 🔧 **Fork 扩展 (可选阅读)**: master 在 `MiniMindModel.__init__` (行~222-228) 额外创建了 `ple_table` / `ple_model_proj` / `ple_proj_norm`, 仅 `use_ple=True` 时存在。forward 中的 PLE 构造段 (行~244-250) 也被 `if self.config.use_ple:` 守卫。**默认 `use_ple=False` 时这些代码完全不执行**, forward 退化为上游原版的 `embed → blocks → norm` 流。

In [ ]:
# 查看 8 个 block
layers = model.model.layers
print(f"层数: {len(layers)}")                     # 8
print(f"类型: {type(layers).__name__}")           # ModuleList

# 每个 block 是 MiniMindBlock
for i, block in enumerate(layers):
    print(f"  layer {i}: {type(block).__name__}, "
          f"attn={type(block.self_attn).__name__}, "
          f"mlp={type(block.mlp).__name__}")

# 验证:数据穿过 block 后 shape 不变
dummy = torch.randn(1, 4, 768)  # (batch, seq_len, hidden_size)
pos_emb = (model.model.freqs_cos[0:4], model.model.freqs_sin[0:4])
with torch.no_grad():
    out, _ = layers[0](dummy, pos_emb)
print(f"\nblock 输入: {dummy.shape}")  # (1, 4, 768)
print(f"block 输出: {out.shape}")      # (1, 4, 768) — shape 不变!

### RoPE buffer 注册

`MiniMindModel.__init__` 的最后几行(219-221)注册了 RoPE(旋转位置编码)的预计算表:

```python
# model_minimind.py:219-221
freqs_cos, freqs_sin = precompute_freqs_cis(
    dim=config.head_dim,               # 96
    end=config.max_position_embeddings, # 32768
    rope_base=config.rope_theta,       # 1e6
    rope_scaling=config.rope_scaling,  # RoPE 缩放配置
)
self.register_buffer("freqs_cos", freqs_cos, persistent=False)
self.register_buffer("freqs_sin", freqs_sin, persistent=False)
```

**`register_buffer` 是什么?**

PyTorch 模块有三种「张量属性」:
- `nn.Parameter` —— 可学习权重,被 optimizer 更新(如 Linear 的 weight)
- `register_buffer` —— **不可学习的持久张量**,随模型一起移动到 GPU(如 RoPE 的 cos/sin 表)
- 普通属性 —— 纯 Python 对象,不随设备迁移

**`persistent=False` 的含义**:这个 buffer **不会出现在 `state_dict()` 里**。因为 RoPE 的 cos/sin 表是可以从 config 参数重新算出来的,不需要存到权重文件里 —— 节省磁盘空间。

> 这也是为什么 forward 里有一段「重新计算」逻辑(239-242 行):用 `init_empty_weights()`(meta device)加载模型时,非 persistent buffer 会丢失,需要检测并重建。

In [ ]:
# 观察 RoPE buffer
print(f"freqs_cos shape: {model.model.freqs_cos.shape}")  # (32768, 96)
print(f"freqs_sin shape: {model.model.freqs_sin.shape}")  # (32768, 96)
print(f"persistent: False（不在 state_dict 里）")

# 验证:不在 state_dict 中
sd = model.state_dict()
print(f"\nstate_dict 中的 buffer keys (含 'freqs'):")
freq_keys = [k for k in sd.keys() if 'freqs' in k]
print(f"  {freq_keys}")  # [] — 空列表,因为 persistent=False

上表中 `freqs_cos` / `freqs_sin` 不在 state_dict 里 —— 这就是 `persistent=False` 的效果。每次加载模型时,它们从 config 参数重新计算。

现在看 forward 怎么用这些 buffer 来定位新 token 的位置:

&nbsp;

---

## 6.3 forward 的 start_pos 计算:RoPE 窗口的起点

`MiniMindModel.forward` 有一个关键设计:**从 KV cache 的长度推算 `start_pos`**,然后切出正确的 RoPE 窗口。

```python
# model_minimind.py:233-261 — MiniMindModel.forward (省略 RoPE 重算分支 239-242)
def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False, **kwargs):
    batch_size, seq_length = input_ids.shape                    # (1, 4)
    past_key_values = past_key_values or [None] * len(self.layers)

    # ★ 关键:start_pos 从上一轮 KV cache 的长度推算
    start_pos = past_key_values[0][0].shape[1] if past_key_values[0] is not None else 0

    hidden_states = self.dropout(self.embed_tokens(input_ids))  # (1, 4, 768)

    # ★ 切出正确位置的 RoPE 表
    position_embeddings = (
        self.freqs_cos[start_pos : start_pos + seq_length],     # (4, 96)
        self.freqs_sin[start_pos : start_pos + seq_length],
    )

    # PLE 构造段 (244-250):默认 use_ple=False 时 ple 恒为 None,整段跳过
    ple = None
    if self.config.use_ple:
        ple = self.ple_model_proj(hidden_states) * (self.config.hidden_size ** -0.5)
        ple = self.ple_proj_norm(ple.view(batch_size, seq_length, self.num_hidden_layers, self.config.ple_dim))
        table = self.ple_table(input_ids).view(batch_size, seq_length, self.num_hidden_layers, self.config.ple_dim)
        ple = (ple + table * (self.config.ple_dim ** 0.5)) * (2 ** -0.5)

    presents = []
    for i, layer in enumerate(self.layers):
        hidden_states, present = layer(
            hidden_states, position_embeddings,
            past_key_value=past_key_values[i],
            use_cache=use_cache,
            attention_mask=attention_mask,
            ple=None if ple is None else ple[:, :, i],  # 按层切片注入 PLE
        )
        presents.append(present)
```

**为什么需要 start_pos?**

RoPE 是位置编码 —— 位置 0 和位置 5 的旋转角度不同。第一次推理时,start_pos=0,用 `freqs_cos[0:4]`。如果用 KV cache 续写,上一轮已经处理了 4 个 token,新 token 的位置是 4,所以用 `freqs_cos[4:5]`。

用代码演示这个过程:

> **循环改写说明**: 当 `ple=None` (默认, `use_ple=False`) 时, 这个 `enumerate` 循环等价于更简洁的 `for layer, pkv in zip(self.layers, past_key_values):` —— `enumerate` + `ple[:,:,i]` 切片仅为支持 PLE 按层注入。本教程后续默认 `use_ple=False`。


In [ ]:
# 模拟两轮推理,展示 start_pos 如何变化

input_ids = torch.tensor([[1, 5310, 2863, 2]])  # (1, 4)

# === 第一轮:没有 KV cache,start_pos = 0 ===
with torch.no_grad():
    hidden, kvs, aux = model.model(input_ids, use_cache=True)

seq_len = input_ids.shape[1]                            # 4
start_pos = kvs[0][0].shape[1]                          # KV cache 第 0 层的 k 张量 seq_len 维度
print(f"【第一轮】")
print(f"  input_ids shape  : {input_ids.shape}")        # (1, 4)
print(f"  seq_length       : {seq_len}")                # 4
print(f"  start_pos        : 0 （无 KV cache）")
print(f"  RoPE 窗口        : freqs_cos[0:4]")
print(f"  RoPE 窗口 shape  : {model.model.freqs_cos[0:4].shape}")  # (4, 96)
print(f"  KV cache k shape : {kvs[0][0].shape}")        # (1, 4, 4, 96) — (batch, seq, n_kv_heads, head_dim)
print(f"  → 下次 start_pos = {start_pos}")

# === 第二轮:用 KV cache 续写 1 个 token,start_pos = 4 ===
new_token = torch.tensor([[5310]])  # (1, 1)

with torch.no_grad():
    hidden2, kvs2, aux2 = model.model(new_token, past_key_values=kvs, use_cache=True)

print(f"\n【第二轮】")
print(f"  input_ids shape  : {new_token.shape}")         # (1, 1)
print(f"  seq_length       : 1")
print(f"  start_pos        : {start_pos} （来自上一轮 KV cache）")
print(f"  RoPE 窗口        : freqs_cos[{start_pos}:{start_pos+1}]")
print(f"  RoPE 窗口 shape  : {model.model.freqs_cos[start_pos:start_pos+1].shape}")  # (1, 96)
print(f"  KV cache k shape : {kvs2[0][0].shape}")        # (1, 5, 4, 96) — seq 从 4→5

可以看到 KV cache 在第二轮从 `(1, 4, 4, 96)` 变成了 `(1, 5, 4, 96)` —— 旧的 4 个 token 的 K/V 被缓存,新 token 的 K/V 被拼接到末尾。start_pos 正确地从 0 跳到了 4。

> 第 7 章会讲 generate 循环如何利用这个机制实现高效的自回归生成。这里只需理解:**start_pos 让 RoPE 能正确定位新 token 的位置**。

&nbsp;

---

## 6.4 最终 RMSNorm:最后一层 block 之后

8 个 block 跑完之后,hidden_states 还要过一次 RMSNorm(262 行):

```python
# model_minimind.py:262
hidden_states = self.norm(hidden_states)
```

这是 `MiniMindModel.__init__` 里创建的(218 行):

```python
self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
```

每个 block 内部都有两个 RMSNorm(input_layernorm 和 post_attention_layernorm),但模型级别的**最终 norm 只有一个** —— 在所有 block 之后、lm_head 之前。

> RMSNorm 的原理在第 4 章已详解。这里只看它在流水线中的位置。

In [ ]:
# 观察 final norm
final_norm = model.model.norm
print(f"类型: {type(final_norm).__name__}")  # RMSNorm
print(f"weight shape: {final_norm.weight.shape}")  # (768,)
print(f"参数量: {final_norm.weight.numel()}")       # 768

# 验证:输入 (1, 4, 768) → 输出 (1, 4, 768)
dummy = torch.randn(1, 4, 768)
with torch.no_grad():
    normed = final_norm(dummy)
print(f"\nnorm 输入: {dummy.shape}")   # (1, 4, 768)
print(f"norm 输出: {normed.shape}")     # (1, 4, 768)
# shape 不变,RMSNorm 只改数值(归一化),不改维度

&nbsp;

---

## 6.5 lm_head:从 hidden_state 到 logits

`MiniMindForCausalLM`(不是 MiniMindModel)在 backbone 上加了一个线性层:

```python
# model_minimind.py:273
self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
#                                768                   6400
```

它把 768 维的 hidden_state 映射到 6400 维的 logits —— 每一维对应词表里的一个 token。

```python
# model_minimind.py:305
logits = self.lm_head(hidden_states[:, slice_indices, :])
```

> `slice_indices` 和 `logits_to_keep` 参数有关(6.8 节提及),默认情况下 `logits_to_keep=0` 表示保留全部位置。

In [ ]:
lm_head = model.lm_head
print(f"类型: {type(lm_head).__name__}")       # Linear
print(f"in_features: {lm_head.in_features}")   # 768
print(f"out_features: {lm_head.out_features}") # 6400
print(f"weight shape: {lm_head.weight.shape}") # (6400, 768)
print(f"bias: {lm_head.bias}")                 # None (bias=False)

# 模拟 hidden_state → logits
dummy_hidden = torch.randn(1, 4, 768)  # backbone 的输出
with torch.no_grad():
    logits = lm_head(dummy_hidden)

print(f"\n输入 (hidden_state): {dummy_hidden.shape}")  # (1, 4, 768)
print(f"输出 (logits):       {logits.shape}")          # (1, 4, 6400)
# (1, 4, 768) → (1, 4, 6400):每个位置对 6400 个 token 各打一个分

关键 shape 变化:

```
hidden_state  (batch, seq_len, hidden_size) = (1, 4, 768)
    │ lm_head: Linear(768 → 6400)
    ▼
logits        (batch, seq_len, vocab_size) = (1, 4, 6400)
```

**logits 是什么?**

logits[t][v] = 模型认为「位置 t 的下一个 token 是词表中第 v 个 token」的**原始分数**(未归一化)。分数越高,模型越倾向于选这个词。

训练时用 softmax + cross_entropy 把 logits 变成概率并计算 loss;推理时用 softmax + 采样(temperature / top_p)从 logits 选出下一个 token(第 7 章)。

在 6.1-6.5 中,我们走完了从 token id 到 logits 的完整路径。接下来看一个关键设计决策:**权重绑定**。

&nbsp;

---

## 6.6 权重绑定(Tied Embeddings):省 490 万参数

这是 minimind 的一个重要设计决策 —— **输入 embedding 和输出 lm_head 共享同一套权重**。

```python
# model_minimind.py:268, 274
_tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}

if self.config.tie_word_embeddings:
    self.model.embed_tokens.weight = self.lm_head.weight  # ← 同一个 Parameter 对象!
```

### 为什么绑定?

`embed_tokens` 把 token id → 768 维向量(编码),`lm_head` 把 768 维向量 → 6400 维分数(解码)。

两者都在同一个 `768 × 6400` 的语义空间里工作 —— 一个是「查表」,另一个是「查表的逆操作」。**共享权重既省参数,又有正则化效果**(避免编码器和解码器学到不一致的表示)。

**省了多少?** `6400 × 768 = 4,915,200` ≈ 490 万参数。在 64M 模型上,这占了 **7.7%** 的总参数量。

In [ ]:
# 验证权重绑定
print("=== 权重绑定验证 ===")
print(f"config.tie_word_embeddings: {config.tie_word_embeddings}")  # True

# embed_tokens 和 lm_head 是同一个对象?
is_tied = model.model.embed_tokens.weight is model.lm_head.weight
print(f"embed_tokens.weight is lm_head.weight: {is_tied}")  # True!

# 它们的 data_ptr 完全相同(同一块内存)
print(f"embed_tokens data_ptr: {model.model.embed_tokens.weight.data_ptr()}")
print(f"lm_head     data_ptr: {model.lm_head.weight.data_ptr()}")
print(f"指针相同: {model.model.embed_tokens.weight.data_ptr() == model.lm_head.weight.data_ptr()}")

# 修改一个,另一个跟着变(因为是同一个对象)
print(f"\n修改前 embed_tokens[0, 0]: {model.model.embed_tokens.weight[0, 0].item():.6f}")
model.lm_head.weight.data[0, 0] = 42.0  # 通过 lm_head 修改
print(f"修改后 embed_tokens[0, 0]: {model.model.embed_tokens.weight[0, 0].item():.6f}")  # 42.0!

# 恢复
model.lm_head.weight.data[0, 0] = model.model.embed_tokens.weight.data[0, 0]

### `_tied_weights_keys` 机制

```python
# model_minimind.py:268
_tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}
```

这个类变量告诉 HuggingFace 的 `PreTrainedModel.post_init()` / `tie_weights()`:**这两个权重应该绑定**。

当用 `from_pretrained()` 加载模型时,HF 框架会:
1. 读取 checkpoint 中的 `lm_head.weight`(或 `model.embed_tokens.weight`)
2. 根据 `_tied_weights_keys` 自动把两者指向同一个 Parameter
3. 这样即使 checkpoint 里只存了一份,加载后两个属性都有效

> 如果 `tie_word_embeddings=False`,`__init__` 里的 `if` 不执行,lm_head 会有独立的随机权重,`_tied_weights_keys` 的绑定在 `tie_weights()` 中会被跳过。

### 对比:不绑定会怎样?

In [ ]:
# 对比 tied vs untied
config_tied = MiniMindConfig(tie_word_embeddings=True)
config_untied = MiniMindConfig(tie_word_embeddings=False)

model_tied = MiniMindForCausalLM(config_tied).eval()
model_untied = MiniMindForCausalLM(config_untied).eval()

n_tied = sum(p.numel() for p in model_tied.parameters())
n_untied = sum(p.numel() for p in model_untied.parameters())

print(f"tied 参数量  : {n_tied:>12,}  ({n_tied/1e6:.2f}M)")
print(f"untied 参数量: {n_untied:>12,}  ({n_untied/1e6:.2f}M)")
print(f"差异        : {n_untied - n_tied:>12,}  ({(n_untied - n_tied) / n_tied * 100:.1f}%)")
# 差异: 4,915,200 (7.7%)

print(f"\ntied: embed is lm_head → {model_tied.model.embed_tokens.weight is model_tied.lm_head.weight}")
print(f"untied: embed is lm_head → {model_untied.model.embed_tokens.weight is model_untied.lm_head.weight}")

可以看到:不绑定时多出 **4,915,200 个参数(7.7%)** —— 这恰好是 `vocab_size × hidden_size` = `6400 × 768`。

在小模型上 7.7% 的参数节省是显著的;在 70B+ 的大模型上则无关紧要 —— 所以大模型通常不绑定,让 lm_head 有独立的表达能力。

&nbsp;

---

## 6.7 完整 forward pass:从 input_ids 到 logits 到 loss

现在把所有零件串起来。下面这段代码**逐步追踪**一个完整 forward pass 中每个张量的 shape:

```
input_ids (1, 4)
    │
    ▼ embed_tokens ──────────── (1, 4, 768)    ← 6.1
    │
    ▼ 8× MiniMindBlock ──────── (1, 4, 768)    ← 6.2 + 6.3
    │
    ▼ final RMSNorm ─────────── (1, 4, 768)    ← 6.4
    │
    ▼ lm_head ───────────────── (1, 4, 6400)   ← 6.5
    │
    ▼ shift + CE loss ───────── scalar          ← 6.8
```

In [ ]:
# === 手动逐步追踪 forward pass ===
input_ids = torch.tensor([[1, 5310, 2863, 2]])  # (1, 4) 假 token 序列
labels = torch.tensor([[5310, 2863, 2, 0]])      # (1, 4) 假标签(右移一位)

print("=" * 60)
print("逐步追踪 forward pass")
print("=" * 60)

# --- Step 1: embed_tokens ---
hidden = model.model.embed_tokens(input_ids)
print(f"\n[1] embed_tokens")
print(f"    input_ids : {input_ids.shape}  → {hidden.shape}")
# (1, 4) → (1, 4, 768): token id 查表变向量

# --- Step 2: dropout (eval 模式下 no-op) ---
hidden = model.model.dropout(hidden)
print(f"\n[2] dropout (eval=no-op)")
print(f"    shape 不变: {hidden.shape}")

# --- Step 3: 切 RoPE 窗口 ---
start_pos = 0
seq_len = input_ids.shape[1]
pos_emb = (model.model.freqs_cos[start_pos:start_pos+seq_len],
           model.model.freqs_sin[start_pos:start_pos+seq_len])
print(f"\n[3] position_embeddings")
print(f"    freqs_cos[{start_pos}:{start_pos+seq_len}] → {pos_emb[0].shape}")
# (4, 96)

# --- Step 4: 8 个 Block ---
print(f"\n[4] 8× MiniMindBlock")
for i, layer in enumerate(model.model.layers):
    with torch.no_grad():
        hidden, present = layer(hidden, pos_emb)
    print(f"    block {i}: {hidden.shape}  (shape 不变)")
# (1, 4, 768) → (1, 4, 768) ×8

# --- Step 5: final RMSNorm ---
hidden = model.model.norm(hidden)
print(f"\n[5] final RMSNorm")
print(f"    shape 不变: {hidden.shape}")
# (1, 4, 768)

# --- Step 6: lm_head ---
logits = model.lm_head(hidden)
print(f"\n[6] lm_head")
print(f"    {hidden.shape} → {logits.shape}")
# (1, 4, 768) → (1, 4, 6400)

上面手动走完了 backbone → lm_head 的 6 步。现在看 loss 怎么算,以及用 `model()` 一步到位是否一致:

In [ ]:
# --- Step 7: CE Loss ---
# 训练时:把 logits 和 labels 右移一位对齐
x = logits[..., :-1, :].contiguous()  # (1, 3, 6400) — 去掉最后一个位置
y = labels[..., 1:].contiguous()       # (1, 3) — 去掉第一个位置

print(f"[7] CE Loss (shift by 1)")
print(f"    logits[:, :-1] : {x.shape}")  # (1, 3, 6400)
print(f"    labels[:, 1:]  : {y.shape}")  # (1, 3)
print(f"    含义: 位置 t 的 hidden → 预测位置 t+1 的 token")

loss = torch.nn.functional.cross_entropy(
    x.view(-1, x.size(-1)),  # (3, 6400)
    y.view(-1),               # (3,)
    ignore_index=-100,
)
print(f"    loss = {loss.item():.4f}")

# === 用 model() 一步完成,验证结果一致 ===
with torch.no_grad():
    output = model(input_ids, labels=labels)

print(f"\n{'=' * 60}")
print(f"model(input_ids, labels) 一步到位:")
print(f"  loss       = {output.loss.item():.4f}")
print(f"  logits     = {output.logits.shape}")
print(f"  aux_loss   = {output.aux_loss.item()}")
print(f"  hidden     = {output.hidden_states.shape}")
print(f"  past_kvs   = {len(output.past_key_values)} 层")

两个 loss 值一致(都是随机初始化下的 ~9.0),说明手动追踪和 `model()` 一步到位完全等价。

### shift by 1 的直觉

为什么 logits 要 `[:, :-1]`,labels 要 `[:, 1:]`?

因为语言模型的任务是**预测下一个 token**:

```
位置:    0      1      2      3
token: [bos]  [你好]  [,]   [eos]
         ↓      ↓      ↓
预测:  [你好]  [,]   [eos]   ???
```

位置 0 的 hidden_state(bos 的表示)应该能预测出位置 1 的 token(你好)。所以 `logits[0]` 对齐 `labels[1]`,`logits[1]` 对齐 `labels[2]`,以此类推 —— 这就是 shift by 1。

&nbsp;

---

## 6.8 CE Loss 与 -100 masking

`MiniMindForCausalLM.forward` 的 loss 计算(307-309 行):

```python
# model_minimind.py:307-309
if labels is not None:
    x = logits[..., :-1, :].contiguous()  # (b, T-1, V)
    y = labels[..., 1:].contiguous()       # (b, T-1)
    loss = F.cross_entropy(x.view(-1, x.size(-1)), y.view(-1), ignore_index=-100)
```

### 为什么用 `ignore_index=-100`?

训练数据中,有些位置的 label 会被设成 `-100`(padding token、或不该学习的 prompt token)。`F.cross_entropy` 的 `ignore_index=-100` 参数让这些位置**不参与 loss 计算** —— 它们的梯度为 0,模型不会从这些位置学习。

这在 SFT 中尤其重要:训练 assistant 模型时,用户说的部分(prompt)被标成 -100,模型只从 assistant 的回复部分学习(第 9 章详解)。

In [ ]:
# 演示 -100 masking 的效果
import torch.nn.functional as F

# 模拟 4 个位置:其中位置 0 和 2 是 padding/prompt(标为 -100)
fake_logits = torch.randn(4, 6400)   # (4, 6400) — 4 个位置的 logits
fake_labels = torch.tensor([-100, 5310, -100, 2])  # 只有位置 1 和 3 参与计算

# 有 ignore_index=-100
loss_masked = F.cross_entropy(fake_logits, fake_labels, ignore_index=-100)
print(f"有 ignore_index=-100 的 loss: {loss_masked.item():.4f}")
print(f"  参与计算的 token 数: {(fake_labels != -100).sum().item()}")  # 2

# 对比:如果把 -100 当普通 label(第 6399 个 token)
fake_labels_no_mask = fake_labels.clone()
fake_labels_no_mask[fake_labels_no_mask == -100] = 0
loss_no_mask = F.cross_entropy(fake_logits, fake_labels_no_mask)
print(f"\n无 masking 的 loss:    {loss_no_mask.item():.4f}")
print(f"  参与计算的 token 数: {len(fake_labels_no_mask)}")  # 4
print(f"\n→ masking 让 loss 只从 assistant 的 token 学习,不受 padding 干扰")

### `logits_to_keep` 参数

```python
# model_minimind.py:304-305
slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) else logits_to_keep
logits = self.lm_head(hidden_states[:, slice_indices, :])
```

默认 `logits_to_keep=0` —— `slice(-0, None)` = `slice(0, None)`,即**保留所有位置**。

当 `logits_to_keep > 0` 时,只保留最后 N 个位置的 hidden_state 来算 logits —— **跳过不需要的 lm_head 计算**。

这在 **RL 训练**(第 13 章 PPO/GRPO)中很有用:采样时你只需要最后一个位置的 logits 来选下一个 token,不需要对所有位置都做 `768→6400` 的投影。

> SFT 不需要它,因为 SFT 的 CE loss 需要所有位置的 logits。第 13 章会详解这个优化。

&nbsp;

---

## 6.9 MoE aux_loss 累加

`MiniMindModel.forward` 的倒数第二行(263 行)有一段关于 MoE 的逻辑:

```python
# model_minimind.py:263
aux_loss = sum(
    [l.mlp.aux_loss for l in self.layers if isinstance(l.mlp, MOEFeedForward)],
    hidden_states.new_zeros(1).squeeze()
)
```

当 `use_moe=True` 时,每个 block 的 FFN 被替换成 `MOEFeedForward`(第 5 章和第 15 章详解)。MoE 需要一个 **auxiliary loss(辅助损失)** 来保证路由器(router)均匀地把 token 分配给各个 expert —— 否则模型会「偷懒」只用一两个 expert。

这段代码做的事:
1. 遍历所有 block,过滤出 MoE 层
2. 把它们的 `aux_loss` 求和
3. 如果没有 MoE 层(默认情况),返回 0

最终这个 `aux_loss` 被包含在 `MoeCausalLMOutputWithPast` 里返回。

下面用代码对比密集模型和 MoE 模型的 `aux_loss` 差异:

In [ ]:
# === 密集模型(默认 use_moe=False)===
print("=== 密集模型 (use_moe=False) ===")
print(f"FFN 类型: {type(model.model.layers[0].mlp).__name__}")  # FeedForward

input_ids = torch.tensor([[1, 5310, 2863, 2]])
with torch.no_grad():
    output = model(input_ids)
print(f"aux_loss: {output.aux_loss.item()}")  # 0.0 — 没有 MoE

# === MoE 模型 (use_moe=True) ===
print(f"\n=== MoE 模型 (use_moe=True) ===")
config_moe = MiniMindConfig(use_moe=True)
model_moe = MiniMindForCausalLM(config_moe).eval()

print(f"FFN 类型: {type(model_moe.model.layers[0].mlp).__name__}")  # MOEFeedForward
print(f"num_experts: {config_moe.num_experts}")                     # 4
print(f"num_experts_per_tok: {config_moe.num_experts_per_tok}")     # 1

# eval 模式下 aux_loss 也为 0(只在 training 模式计算)
with torch.no_grad():
    output_moe = model_moe(input_ids)
print(f"aux_loss (eval): {output_moe.aux_loss.item()}")

# train 模式下才有非零 aux_loss
model_moe.train()
with torch.no_grad():
    output_moe_train = model_moe(input_ids)
print(f"aux_loss (train): {output_moe_train.aux_loss.item():.6f}")
# 非 0 — 4 个 expert 的负载均衡损失之和

> MoE 的 aux_loss 会在训练循环中被加到主 loss 上:`total_loss = ce_loss + aux_loss`。权重由 `router_aux_loss_coef`(默认 5e-4)控制,非常小,起正则化作用。第 15 章会深入 MoE 的路由机制。

&nbsp;

---

## 6.10 参数量验证:拆解 64M 模型

最后,把整个模型的参数量拆开看,验证我们的理解是否正确。

前 9 节我们走完了完整的 forward pass —— 从 input_ids 到 loss。现在回头算一笔账:这台引擎到底有多少「零件」(参数)?

In [ ]:
# === 完整参数量拆解 ===
model = MiniMindForCausalLM(MiniMindConfig()).eval()
total = sum(p.numel() for p in model.parameters())

print("=" * 55)
print(f"{'组件':<25} {'参数量':>12}  {'占比':>7}")
print("=" * 55)

# 1. Embedding
embed = model.model.embed_tokens.weight.numel()
print(f"{'embed_tokens (tied)':<25} {embed:>12,}  {embed/total*100:>6.1f}%")

# 2. Per block breakdown
b = model.model.layers[0]
attn = sum(p.numel() for p in b.self_attn.parameters())
ffn = sum(p.numel() for p in b.mlp.parameters())
norms = b.input_layernorm.weight.numel() + b.post_attention_layernorm.weight.numel()
per_block = attn + ffn + norms

print(f"{'  attention (per block)':<25} {attn:>12,}")
print(f"{'  ffn (per block)':<25} {ffn:>12,}")
print(f"{'  norms (per block)':<25} {norms:>12,}")
print(f"{'  ── 1 block 合计':<25} {per_block:>12,}")
print(f"{'  × 8 blocks':<25} {per_block*8:>12,}  {per_block*8/total*100:>6.1f}%")

# 3. Final norm
final_norm = model.model.norm.weight.numel()
print(f"{'final RMSNorm':<25} {final_norm:>12,}")

# 4. lm_head (tied → 0 额外参数)
print(f"{'lm_head (tied=0)':<25} {'0':>12}")

# 5. 总计
manual_total = embed + per_block * 8 + final_norm
print("-" * 55)
print(f"{'手动合计':<25} {manual_total:>12,}")
print(f"{'model.parameters()':<25} {total:>12,}")
print(f"{'差异':<25} {total - manual_total:>12}  ← 应为 0")

print(f"\n总参数量: {total:,} ({total/1e6:.2f}M)")

手动计算和 `model.parameters()` 的结果完全吻合 —— **63,912,192 ≈ 63.9M**。

### 参数分布

| 组件 | 参数量 | 占比 |
|---|---|---|
| embed_tokens | 4,915,200 | 7.7% |
| 8× Attention | 14,157,312 | 22.2% |
| 8× FeedForward | 44,826,624 | 70.1% |
| 8× Norms + final | 13,056 | 0.0% |
| **总计** | **63,912,192** | **100%** |

可以看到 **FFN 是参数大户(70%)**,Attention 占 22%,Embedding 占 8%。这是现代 Transformer 的典型分布 —— 模型越大,FFN 占比越高。

> 如果 `tie_word_embeddings=False`,lm_head 多 4,915,200 参数,总参数量变成 **68.8M**,embedding+lm_head 合计占 14.3%。

### forward 的返回类型

### MoeCausalLMOutputWithPast:forward 返回什么

`MiniMindForCausalLM.forward` 返回一个 `MoeCausalLMOutputWithPast`(来自 HuggingFace transformers):

```python
# model_minimind.py:310
return MoeCausalLMOutputWithPast(
    loss=loss,                  # CE loss（或 None 如果没传 labels）
    aux_loss=aux_loss,          # MoE 辅助损失（密集模型为 0）
    logits=logits,              # (b, T, vocab_size) 预测分数
    past_key_values=past_kvs,   # KV cache（推理时复用）
    hidden_states=hidden_states # (b, T, hidden_size) backbone 输出
)
```

这个返回值同时满足训练(需要 loss)和推理(需要 logits + past_key_values)的需求。

> 为什么用 MoE 版本的输出类?因为 minimind 的密集模型和 MoE 模型共用同一个 forward,`MoeCausalLMOutputWithPast` 包含 `aux_loss` 字段,密集模型返回 0 即可。

&nbsp;

---

## Summary and takeaways

本章我们把第 3-5 章学到的零件**组装成了完整的模型**。核心要点:

### 1. MiniMindModel(backbone)的职责

```
input_ids (b, T)
  → embed_tokens → (b, T, 768)      ← token id 查表变向量
  → 8× Block     → (b, T, 768)      ← 逐层提取语义
  → final norm   → (b, T, 768)      ← 归一化
```

backbone 负责「编码」:把离散的 token id 序列编码成连续的 hidden_state 序列。

### 2. MiniMindForCausalLM 的职责

在 backbone 上加了:
- **lm_head**: `(b, T, 768) → (b, T, 6400)`,把 hidden_state 映射到词表空间
- **CE loss**: shift by 1 + `ignore_index=-100`,只从有效 token 学习
- **aux_loss**: MoE 的负载均衡损失(密集模型为 0)

### 3. 权重绑定

`embed_tokens.weight is lm_head.weight` —— 同一块内存。省 7.7% 参数 + 正则化。

### 4. RoPE buffer + start_pos

`register_buffer(persistent=False)` 存预计算的 cos/sin 表;`start_pos` 从 KV cache 长度推算,切正确位置的 RoPE 窗口。

### 5. 整条流水线的 shape

| 步骤 | shape 变化 |
|---|---|
| input_ids | `(b, T)` |
| embed_tokens | `(b, T, 768)` |
| 8× Block | `(b, T, 768)` (不变) |
| final norm | `(b, T, 768)` (不变) |
| lm_head | `(b, T, 6400)` |
| CE loss | scalar |

---

> **下一步**:现在我们有了 logits —— 模型对每个位置下一个 token 的打分。但 logits 还不是文字。怎么从 logits 选出下一个 token?怎么高效地一个一个 token 往后生成?这就是第 7 章的主题:**生成 —— KV cache + 采样循环**。
>
> → [第 7 章 · 生成:logits 怎么变成文本](../ch07/01_main-chapter-code/README.md)